# SONA — Complex Ratio Mask vs Magnitude Mask (Full VoiceBank-824)

Promotes the local pilot (complex mask +0.32 dB SI-SDR on a 1200-clip subset) to a
**full-corpus, PESQ-backed** run so it is publishable.

Trains two GTCRN denoisers that are **identical except for the output representation** —
`magnitude_gtcrn` (real per-band gain, noisy phase preserved) vs `complex_gtcrn` (per-band
complex ratio mask, modifies magnitude **and** phase) — under the same data, epochs, loss,
optimiser and backbone width. Evaluates both on the full 824-clip test set with the full
metric suite **including PESQ**, and writes `per_sample.csv` so paired significance tests run.

**Requires:** GPU accelerator ON, the refreshed `sona.zip` (must contain
`scripts/complex_mask_full.py`), and a VoiceBank-DEMAND dataset with the standard
`clean_trainset_wav / noisy_trainset_wav / clean_testset_wav / noisy_testset_wav` layout.


In [ ]:
%pip install -q pesq pystoi

import torch, torchaudio
print('torch', torch.__version__, '| torchaudio', torchaudio.__version__)
print('cuda available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Turn on the GPU accelerator before running.'

In [ ]:
import glob
from pathlib import Path

def find_repo():
    # freshness guard: the uploaded sona.zip MUST contain the new script
    hits = glob.glob('/kaggle/input/**/scripts/complex_mask_full.py', recursive=True)
    hits = sorted(hits, key=lambda p: ('sona' not in p.lower(), len(p)))
    return str(Path(hits[0]).parents[1]) if hits else None

def has_vbd_layout(root):
    root = Path(root)
    needed = ['clean_trainset_wav', 'noisy_trainset_wav', 'clean_testset_wav', 'noisy_testset_wav']
    return all((root / n).is_dir() for n in needed)

def find_vbd():
    hits = glob.glob('/kaggle/input/**/noisy_trainset_wav', recursive=True)
    roots = [str(Path(h).parent) for h in hits if has_vbd_layout(Path(h).parent)]
    return sorted(roots, key=len)[0] if roots else None

REPO_DIR = find_repo()
VBD_ROOT = find_vbd()
OUT_DIR = '/kaggle/working/results_complex_full824'

# If auto-detection fails, set manually from the Kaggle right-hand panel:
# REPO_DIR = '/kaggle/input/sona-new'
# VBD_ROOT = '/kaggle/input/voicebank-demand-16k'

assert REPO_DIR, 'sona repo with scripts/complex_mask_full.py not found — upload the refreshed sona.zip.'
assert VBD_ROOT, 'VoiceBank-DEMAND (train+test wav folders) not found.'
assert (Path(REPO_DIR) / 'src' / 'models' / 'gtcrn.py').exists()
print('REPO_DIR =', REPO_DIR)
print('VBD_ROOT =', VBD_ROOT)
print('OUT_DIR  =', OUT_DIR)

In [ ]:
# Full-corpus A/B config. Both models share this backbone/loss; only the mask type differs.
LIMIT_TRAIN = None    # None = full VoiceBank trainset (~11.5k clips)
LIMIT_TEST  = 824     # full test set
EPOCHS      = 50
CH          = 96
HIDDEN      = 96
N_BLOCKS    = 4
MAG_WEIGHT  = 1.25    # matches the locked Layer A loss
MR_MAG_WEIGHT = 0.25
SNR_WEIGHT  = 0.3     # scale anchor: keeps the complex mask from drifting output level
SEED        = 1234

print(dict(LIMIT_TRAIN=LIMIT_TRAIN, LIMIT_TEST=LIMIT_TEST, EPOCHS=EPOCHS,
           CH=CH, HIDDEN=HIDDEN, N_BLOCKS=N_BLOCKS, MAG_WEIGHT=MAG_WEIGHT,
           MR_MAG_WEIGHT=MR_MAG_WEIGHT, SNR_WEIGHT=SNR_WEIGHT))

In [ ]:
import subprocess, sys

cmd = [
    sys.executable, f'{REPO_DIR}/scripts/complex_mask_full.py',
    '--repo', REPO_DIR,
    '--vbd-root', VBD_ROOT,
    '--out', OUT_DIR,
    '--limit-test', str(LIMIT_TEST),
    '--epochs', str(EPOCHS),
    '--ch', str(CH), '--hidden', str(HIDDEN), '--n-blocks', str(N_BLOCKS),
    '--mag-weight', str(MAG_WEIGHT), '--mr-mag-weight', str(MR_MAG_WEIGHT),
    '--snr-weight', str(SNR_WEIGHT), '--seed', str(SEED),
]
if LIMIT_TRAIN is not None:
    cmd += ['--limit-train', str(LIMIT_TRAIN)]

print(' '.join(cmd), '\n')
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end='')
proc.wait()
assert proc.returncode == 0, f'complex_mask_full.py exited {proc.returncode}'

In [ ]:
import pandas as pd
from pathlib import Path

print(Path(OUT_DIR, 'summary.md').read_text(encoding='utf-8'))
df = pd.read_csv(Path(OUT_DIR, 'summary.csv'))
display(df)

# paired win/loss with PESQ front-and-centre
mag = df[df.method == 'magnitude_gtcrn'].iloc[0]
cx  = df[df.method == 'complex_gtcrn'].iloc[0]
higher = {'si_sdr','si_sdr_i','snr','snr_i','pesq','stoi','estoi'}
print(f"{'metric':9s} {'magnitude':>10s} {'complex':>10s} {'delta':>10s}")
for k in ['si_sdr','si_sdr_i','snr','snr_i','pesq','stoi','estoi','lsd']:
    if k in df.columns and pd.notna(mag[k]) and pd.notna(cx[k]):
        d = cx[k] - mag[k]
        better = (d > 0) if k in higher else (d < 0)
        print(f"{k:9s} {mag[k]:10.4f} {cx[k]:10.4f} {d:+10.4f}  {'<-- complex' if better else ''}")

In [ ]:
# Optional: paired Wilcoxon + bootstrap CIs on the per-clip results (if significance.py is present)
import subprocess, sys
sig = f'{REPO_DIR}/../sona-final-month/scripts/significance.py'
from pathlib import Path
if Path(sig).exists():
    subprocess.run([sys.executable, sig, '--results-dir', OUT_DIR, '--out-dir', OUT_DIR + '/sig'], check=False)
else:
    print('significance.py not co-located; run it locally on per_sample.csv:')
    print('  methods to compare: complex_gtcrn vs magnitude_gtcrn on si_sdr_i / pesq / stoi')

In [ ]:
import glob, os, shutil
from IPython.display import FileLink, display

ART = '/kaggle/working/sona_complex_mask_full824'
shutil.make_archive(ART, 'zip', OUT_DIR)
print(f'Wrote {ART}.zip ({os.path.getsize(ART + ".zip")/1e6:.1f} MB)')
for f in sorted(glob.glob(OUT_DIR + '/**', recursive=True)):
    if os.path.isfile(f):
        print('  ', os.path.relpath(f, OUT_DIR), f'({os.path.getsize(f)/1e3:.0f} KB)')
display(FileLink(ART + '.zip'))